![Generative AI Practical Workshop](../assets/workshop_banner.png)

# Prompt Engineering Lab

Most people jump straight to ChatGPT and type a question. In this session we slow down and ask a different question: **is the prompt good enough to trust?**

You will write prompts, run them, compare answers, and score what comes back. Bring your `.env` file (one API key) or Ollama — either is fine.


### How this fits the full workshop

![Workshop flow](../assets/notebook_flow_diagram.png)

You are on **Notebook 1 (Prompting)**. Notebooks 2 and 3 add Gradio apps, streaming, tools, and multimodal outputs.

## What you will practise

- Summarising, classifying, and reasoning with prompts
- Turning vague prompts into clear instructions
- Comparing weak vs strong prompts on the same task
- Scoring outputs with a simple rubric
- Seeing why **testing** matters more than clever wording


## Step 1: Set up your notebook

1. Make sure you ran `pip install -r requirements.txt` and copied `.env.example` to `.env`.
2. Run the next two cells.
3. Look at the provider dictionary. At least one entry should be `True` (often `ollama` if you run models locally).


In [1]:
import sys
from pathlib import Path
import pandas as pd

# Run this cell first. It finds the project folder and loads your .env file.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [2]:
from src.llm_gateway import check_available_providers, run_llm

SELECTED_PROVIDER = "auto"  # change to "ollama", "openai", etc. if you want
check_available_providers()

{'openai': False,
 'anthropic': False,
 'gemini': False,
 'mistral': False,
 'cohere': False,
 'deepseek': False,
 'groq': False,
 'openrouter': False,
 'ollama': True}

## Step 2: Optional: check Ollama

Skip this if you are using a cloud API key only.

If Ollama is installed, open a terminal and run `ollama serve`. Pull any text model you like (`ollama pull llama3.2` or use one you already have).


In [3]:
from src.llm_gateway import check_ollama_server, resolve_ollama_model

check_ollama_server()
print("Using model:", resolve_ollama_model())

Using model: llava:latest


## Step 3: Start with a weak prompt

We use a short reading about generative AI. First we ask the model in the laziest way possible — on purpose.


In [4]:
from src.utils import read_sample_text

source_text = read_sample_text("generative_ai_intro.txt")
weak_prompt = f"Summarise this text: {source_text[:700]}"

print(weak_prompt[:250], "...")

Summarise this text: Generative AI is a branch of artificial intelligence focused on creating new content such as text, images, audio, code, and video. Instead of only classifying or predicting from existing data, generative systems learn patterns fr ...


In [5]:
from src.output_formatting import show_llm_output

weak_output = run_llm(weak_prompt, provider=SELECTED_PROVIDER)
show_llm_output(weak_output, title="Weak prompt result")

## Weak prompt result

Generative AI refers to a subset of artificial intelligence that focuses on creating new content such as text, images, audio, code, and video by learning patterns from large datasets. One popular example is Large Language Models (LLMs) which are trained on vast collections of text and can perform various tasks like answering questions, summarizing documents, translating languages, and helping with coding tasks. However, the output generated by these models is probabilistic, meaning that they predict likely next words rather than retrieving verified facts from a database.

**Pause and look at the answer.**

- Did it follow a clear structure?
- Did it add facts that are not in the source?
- Would you hand this to a classmate without checking it?

Most weak prompts fail quietly. That is why we test.


## Step 4: Use a clearer prompt pattern

In class we build prompts from six parts:

`Role + Task + Context + Constraints + Output format + Quality check`

You do not need fancy language. You need **specific instructions**.


In [6]:
from src.prompt_templates import build_prompt

strong_prompt = build_prompt(
    role="You are a teaching assistant helping first-year university students.",
    task="Summarise the source material for a practical class.",
    context=f"Source material:\n{source_text[:700]}",
    constraints=(
        "- Use only information from the source.\n"
        "- Use plain language.\n"
        "- Do not invent references or examples that are not supported."
    ),
    output_format=(
        "1. Summary — 2–3 sentences\n"
        "2. Key points — three bullets\n"
        "3. One question a student might ask"
    ),
    quality_check="Could a first-year student use this to revise without being misled?",
)

strong_output = run_llm(strong_prompt, provider=SELECTED_PROVIDER)
show_llm_output(strong_output, title="Strong prompt result")

## Strong prompt result

Summary:
Generative AI is a branch of AI that creates new content using patterns learned from large datasets. Large Language Models (LLMs) are an example, and they can perform various tasks like answering questions or writing drafts, but their outputs are probabilistic rather than factual.

Key points:
* Generative AI focuses on creating new content by learning patterns from data.
* LLMs are a well-known example of generative AI that can perform multiple tasks.
* The outputs generated by LLMs are not always factual and can be probabilistic.

Question:
Why is the output of a large language model sometimes not factual?

## Step 5: Summarisation challenge (compare five versions)

Run the versions below. After each one, give it a human score from 1–5 in the table.


In [7]:
import pandas as pd

prompt_results = []
versions = {
    "1 weak": weak_prompt,
    "2 structured": strong_prompt,
    "3 audience-aware": strong_prompt.replace("first-year", "busy postgraduate"),
    "4 faithfulness": strong_prompt + "\nIf a detail is missing, write: Not in source.",
    "5 json": strong_prompt + "\nReturn JSON with keys: summary, bullets, question.",
}

for name, p in versions.items():
    out = run_llm(p, provider=SELECTED_PROVIDER, max_tokens=700)
    prompt_results.append({
        "Version": name,
        "Human score (1-5)": None,
        "Notes": "",
        "Output preview": out[:200] + "...",
    })

pd.DataFrame(prompt_results)

,Version,Human score (1-5),Notes,Output preview
0,1 weak,None,,Generative AI is an area of artificial intelli...
1,2 structured,None,,Summary:\nGenerative AI is a type of artificia...
2,3 audience-aware,None,,Summary:\nGenerative AI refers to the branch o...
3,4 faithfulness,None,,Summary:\nGenerative AI is a type of artificia...
4,5 json,None,,Summary:\nGenerative AI is an AI branch focuse...


## Step 6: Classification (student support messages)

We will sort messages into: Academic Support, Administrative Support, Technical Support, Wellbeing Concern.

**Important:** for wellbeing messages, the model should flag escalation to a human — not give medical advice.


In [8]:
messages_df = pd.read_csv(PROJECT_ROOT / "data/sample_texts/student_feedback_examples.csv")
messages_df

,message_id,message,expected_label
0,1,I cannot access the lecture slides for module ...,Technical Support
1,2,I am feeling overwhelmed and have not been sle...,Wellbeing Concern
2,3,Could you confirm whether the assignment deadl...,Administrative Support
3,4,I do not understand the difference between pre...,Academic Support
4,5,My laptop crashes whenever I run the Jupyter k...,Technical Support
5,6,I need advice on whether I should change my di...,Academic Support
6,7,I lost my student ID card and need to know the...,Administrative Support
7,8,I have been anxious about presenting in class ...,Wellbeing Concern


In [9]:
from src.prompt_templates import build_prompt

messages_block = "\n".join(
    f"{row['message_id']}. {row['message']}"
    for _, row in messages_df.iterrows()
)

classification_prompt = build_prompt(
    role="You are a university student support triage assistant.",
    task="Classify each student message into exactly one label.",
    context=f"Messages:\n{messages_block}",
    constraints=(
        "Labels (use exactly one per message):\n"
        "- Academic Support: course content, study skills, dissertation advice\n"
        "- Administrative Support: deadlines, ID cards, portal issues\n"
        "- Technical Support: software, hardware, access errors\n"
        "- Wellbeing Concern: stress, anxiety, sleep, mental health\n\n"
        "Wellbeing messages must be escalated to a human support officer. "
        "Do not provide diagnosis, counselling, or medical advice."
    ),
    output_format=(
        "For each message_id, return:\n"
        "- label\n"
        "- one-line reason\n"
        "- escalation note (if Wellbeing Concern, say: escalate to human)"
    ),
    quality_check="Would a student support team trust this routing?",
)

from src.output_formatting import show_llm_output

show_llm_output(
    run_llm(classification_prompt, provider=SELECTED_PROVIDER, max_tokens=900),
    title="Classification task",
)

## Classification task

### Administrative Support
The message discusses an issue with accessing course content through the portal, which is not related to mental health or academic progress. Escalation note: None.

### Wellbeing Concern
The message expresses stress and anxiety about exams, suggesting a concern for mental health. Escalation note: escalate to human.

### Administrative Support
The message inquires about the application of deadline extension policies for students who were ill, which is not related to mental health or academic progress. Escalation note: None.

### Academic Support
The message asks for clarification on the difference between precision and recall in a classification notebook, which relates to academic progress and study skills. Escalation note: None.

### Technical Support
The message reports that the Jupyter kernel crashes when running the neural network practical, which is related to software or hardware issues. Escalation note: None.

### Academic Support
The message seeks advice on changing a dissertation topic due to the supervisor's unavailability, which relates to academic progress and study skills. Escalation note: None.

### Administrative Support
The message reports the loss of a student ID card and inquires about the replacement process and cost, which is not related to mental health or academic progress. Escalation note: None.

### Wellbeing Concern
The message expresses anxiety about presenting in class, suggesting a concern for mental health. Escalation note: escalate to human.

## Step 7: Reasoning (without asking for hidden chain-of-thought)

Scenario: a lecturer considers giving students one extra week before a practical exam.


In [10]:
from src.prompt_templates import build_prompt

reasoning_prompt = build_prompt(
    role="You are an academic advisor helping a lecturer make a fair decision.",
    task="Should the lecturer give students one extra week before a practical exam?",
    context=(
        "Many students missed the revision session because of a public transport strike. "
        "The exam tests practical skills covered in weeks 8–10. "
        "Some students have already submitted practice work early."
    ),
    constraints=(
        "- Do not reveal hidden chain-of-thought.\n"
        "- Base the answer on the scenario only.\n"
        "- Mention students who prepared early as a fairness consideration."
    ),
    output_format=(
        "1. Recommendation\n"
        "2. Assumptions you are making\n"
        "3. Evidence from the scenario\n"
        "4. Risks of your recommendation\n"
        "5. Final decision statement"
    ),
    quality_check="Is the recommendation fair and actionable without overclaiming?",
)

from src.output_formatting import show_llm_output

show_llm_output(
    run_llm(reasoning_prompt, provider=SELECTED_PROVIDER, max_tokens=800),
    title="Reasoning task",
)

## Reasoning task

### Recommendation
Yes, the lecturer should give students one extra week before the practical exam to make up for the missed revision session due to the public transport strike.

### Assumptions
The scenario does not provide information on whether students who prepared early were given any advantages or if there are any other factors that may impact the students' performance in the exam.

### Evidence from the scenario
Many students missed the revision session because of a public transport strike, and some students have already submitted practice work early.

### Risks of the recommendation
Giving students an extra week before the exam could potentially lead to delays in the academic calendar or make it difficult for the lecturer to cover all the necessary material within the given time frame.

### Final decision statement
Considering the public transport strike and the fact that some students have already submitted practice work early, granting students an extra week before the practical exam would be a fair decision.

## Step 8: Score an answer with a rubric

Rate the structured summary (or any output you choose) from 1 to 5 on each row.


In [11]:
from src.prompt_templates import score_output_rubric, RUBRIC_CRITERIA

example_scores = {
    "Clarity": 4,
    "Relevance": 4,
    "Faithfulness": 3,
    "Completeness": 4,
    "Format control": 3,
    "Usefulness": 4,
    "Safety": 5,
}

score_output_rubric(example_scores)


,Criterion,Score (1-5)
0,Clarity,4
1,Relevance,4
2,Faithfulness,3
3,Completeness,4
4,Format control,3
5,Usefulness,4
6,Safety,5


## Step 9: LLM-as-judge (use carefully)

A model can help you critique an answer. It can also be wrong. **Always keep a human in the loop.**


In [12]:
from src.prompt_templates import build_llm_judge_prompt

judge_prompt = build_llm_judge_prompt(
    task="Summarise source material for first-year students",
    prompt_used=strong_prompt[:500] + "...",
    model_output=strong_output[:800] + "...",
)

from src.output_formatting import show_llm_output

show_llm_output(
    run_llm(judge_prompt, provider=SELECTED_PROVIDER, max_tokens=700),
    title="LLM-as-judge critique",
)

## LLM-as-judge critique

Strength: The summary is clear and concise, and it accurately reflects the main points of the source material. It effectively highlights the key aspects of generative AI and LLMs, making it easy for first-year students to understand the topic.

Weakness: The summary does not provide enough detail about the specific tasks that LLMs can perform or how they generate novel outputs. Additionally, it could have been more comprehensive by including more examples of generative AI applications or discussing the limitations and potential risks associated with these systems.

Reminder: It's important to remember that a human must review the output generated by an LLM to ensure its factual accuracy and relevance before using or relying on it.

## Prompt clinic (group activity)

Here are five bad prompts. In your group, say what is wrong and rewrite one of them using the six-part pattern.

1. Make this better.
2. Explain AI.
3. Classify these students as serious or unserious.
4. Think deeply and answer.
5. Summarise this document but make it perfect.


## Wrap-up

The skill is not writing the cleverest prompt. It is **checking whether the prompt works** before you rely on it.

In Notebook 2 you will turn a good prompt into a small app.
